## 統計**離群值**處理：**Z-score 與 IQR 演算法**

- 目標：掌握 Z-score（標準分數）與 IQR（四分位距）兩種經典統計離群值檢測演算法的物理意義。學會將手動判讀的直覺轉化為全自動的資料清洗 Pipeline 程式邏輯，以**過濾製程或測試中的異常雜訊**。


### 1. 離群值檢測的物理意義與演算法挑選

- 核心：
    - 在半導體測試或材料實驗數據中，常**因探針接觸不良、儀器瞬間突波或偶發性材料缺陷**，產生極端的離群值（Outliers）。
    - **Z-score（標準分數法）**：基於資料符合常態分佈的假設。計算每個點偏離均值幾個標準差，通常將 |Z| > 3 的點判定為異常。
    - **IQR（四分位距法 / 盒鬚圖法）**：不需假設資料分佈型態。利用第 75 百分位數（Q3）與第 25 百分位數（Q1）計算出間距，高於 Q3 + 1.5*IQR 或低於 Q1 - 1.5*IQR 即視為異常，防禦髒資料的能力極強。


### 2. Z-score 與 IQR 演算法的程式化流程實作

- 實作：結合材料背景實務，模擬一份**高頻電性量測或材料添加劑（5phr 離群值判讀經驗）的原始數據**。這組數據中包含了偶發性的量測異常值，我們將寫程式自動將其揪出。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

# --- 模擬 60 筆製程量測數據，並故意塞入 3 筆極端的髒資料（離群值）---
np.random.seed(24)
raw_measurements = np.random.normal(loc=15.0, scale=0.8, size=60)
# 塞入離群值
raw_measurements[12] = 22.5  # 突波高值
raw_measurements[35] = 5.2  # 突波低值
raw_measurements[52] = 19.8  # 突波高值

# 建立 DataFrame 方便清洗
df_clean = pd.DataFrame({"measurement": raw_measurements})


# --- 流程 A：Z-score 離群值檢測（對極端值敏感） ---
print(">>> 啟動 Z-score 異常檢測流程...")
# 計算每個點的 Z-score
df_clean["z_score"] = stats.zscore(df_clean["measurement"])

# 定義門檻值（業界標準通常為 3 個標準差）
z_threshold = 3.0
df_clean["is_z_outlier"] = df_clean["z_score"].abs() > z_threshold

z_outliers = df_clean[df_clean["is_z_outlier"]]
print(
    f"Z-score 偵測到 {len(z_outliers)} 筆離群值，索引分別為: {z_outliers.index.tolist()}"
)


# --- 流程 B：IQR 離群值檢測（強健性高） ---
print("\n>>> 啟動 IQR 異常檢測流程...")
Q1 = df_clean["measurement"].quantile(0.25)
Q3 = df_clean["measurement"].quantile(0.75)
IQR = Q3 - Q1

# 計算圍牆界限 (Fence)
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

df_clean["is_iqr_outlier"] = (df_clean["measurement"] < lower_fence) | (
    df_clean["measurement"] > upper_fence
)

iqr_outliers = df_clean[df_clean["is_iqr_outlier"]]
print(
    f"IQR 偵測到 {len(iqr_outliers)} 筆離群值，索引分別為: {iqr_outliers.index.tolist()}"
)
print(f"IQR 圍牆界限：[{lower_fence:.3f} ~ {upper_fence:.3f}]")

### 3. 視覺化對照與資料清洗 (Data Cleaning)

- 抓出離群值後，我們可以用圖表視覺化對照，並將資料進行剔除或抽換。


In [ ]:
# 繪製檢測結果圖表
plt.figure(figsize=(9, 4.5))
plt.plot(
    df_clean["measurement"],
    marker="o",
    color="gray",
    alpha=0.5,
    label="正常量測值",
    linestyle=":",
)

# 標註 IQR 篩選出來的離群值
plt.scatter(
    iqr_outliers.index,
    iqr_outliers["measurement"],
    color="red",
    marker="x",
    s=100,
    zorder=5,
    label="IQR 離群值",
)
plt.axhline(y=upper_fence, color="red", linestyle="--", alpha=0.6, label="IQR 上圍牆")
plt.axhline(y=lower_fence, color="red", linestyle="--", alpha=0.6, label="IQR 下圍牆")

plt.title("自動化離群值過濾與檢測 Pipeline 對照圖", fontsize=12)
plt.xlabel("樣本觀測索引 (Sample Index)")
plt.ylabel("量測數值")
plt.legend()
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

# 資料清洗：剔除異常值，保留乾淨的 DataFrame 用於機器學習模型訓練
print("\n>>> 執行資料清洗作業...")
df_final_clean = df_clean[~df_clean["is_iqr_outlier"]].copy()
print(
    f"原始資料筆數: {len(df_clean)} 筆 -> 清洗後乾淨資料筆數: {len(df_final_clean)} 筆"
)

- 總結：在過去的材料研發或實驗室實務中（例如論文中 5phr 離群值的判讀經驗），過去多半依賴人工直覺或手動拉盒鬚圖來挑出異常點。但在『柔性測試集成工程師』的思維裡，必須將這種直覺轉換為可重複執行的程式化流程。在我的專案中，我同時實作了 Z-score 與 IQR 兩種過濾機制。我更偏好在資料最前端使用 IQR 演算法，因為它利用分位數計算，不會像 Z-score 一樣容易受到極端值本身拉扯均值的影響（即不受遮蔽效應 Masking Effect 干擾）。透過這個自動化清洗腳本，我們能在機台測試資料輸入到 XGBoost 或神經網路預測良率之前，先剔除掉硬體突波引起的髒資料，這對維持後續 ML 模型的預測精度至關重要。
